# Figure 1: Canonical figure reproduction

Regenerate from immutable processed artifacts without loading FlyVis models. Models/seeds are the analysis units; original metric signs are retained. Figure 1B retains cached representative snapshots because model-013 raw vectors are unavailable. Numerical inputs and output hashes are recorded beside each figure.


## 1. Imports and configuration
Artifact-only replay; no models or acquisition Notebook execution.


In [1]:
# Colab needs the repository's saved analysis inputs, not just the package.
import importlib
import importlib.util
import os
import site
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/flyvis-midd")
    if not (ROOT / "src/flyvis_midd").is_dir():
        if ROOT.exists():
            raise RuntimeError("/content/flyvis-midd exists but is not a repository checkout.")
        result = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             "https://github.com/poxsalop5029/flyvis-midd.git", str(ROOT)],
            capture_output=True, text=True,
        )
        if result.returncode:
            message = result.stderr or result.stdout
            raise RuntimeError("Could not clone the public flyvis-midd repository: " + message[-2000:])
    missing = [name for name in ("numpy", "pandas", "matplotlib")
               if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
        for package_dir in site.getsitepackages():
            site.addsitedir(package_dir)
    importlib.invalidate_caches()
else:
    ROOT = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "src/flyvis_midd").is_dir()), None,
    )
    if ROOT is None:
        raise RuntimeError("Run this notebook from inside the flyvis-midd repository.")
sys.path.insert(0, str(ROOT / "src"))
print("Repository inputs:", ROOT)


from IPython.display import display
from flyvis_midd.figure_analysis import (
    input_manifest, load_tables, validate_inputs, primary_analysis,
    adopted_analysis, statistical_tables, write_reproduction_record,
)
from flyvis_midd.figure_rendering import render
FIGURE = '1'


Repository inputs: <repository-root>


## 2. Input artifact manifest
Source paths and SHA-256 identify the immutable numerical inputs. Bundled precomputed artifacts are numerical inputs.


In [2]:
manifest = input_manifest(FIGURE, ROOT)
display(manifest)


,path,sha256,bytes
0,data/precomputed/fig01/fig1_midd_per_model.csv,ea64ba2c8b5d1bee163f70c30901bf6063849957771137...,28361
1,figures/fig01/source/retained_representative_f...,7500e31838899d6df4a7f3e0548a7dc4ee76ac04e24ee2...,369925
2,figures/fig01/fig1A_midd_stimuli.png,d15d282c4eecc20b6e3c72da955329d2f9bef3911ddf53...,192925


## 3. Canonical data load
Each source keeps its cohort, renderer, operator, temporal support, and metric.


In [3]:
tables = load_tables(manifest, ROOT)
print("Loaded source tables:", len(tables))


Loaded source tables: 1


## 4. QA and assertions
Check within-contract A/S algebra and adopted source equivalence before saving.


In [4]:
integration_checks = validate_inputs(tables)
display(integration_checks)


,check,passed
0,A/S identity: data/precomputed/fig01/fig1_midd...,True


## 5. Primary analysis
For Fig3/S7, affine fits and both held-out splits are recomputed from saved A3/A4. Native and multi-field definitions remain separate.


In [5]:
primary = primary_analysis(FIGURE, tables)
for name, frame in primary.items():
    print(name)
    display(frame.head(20))


data/precomputed/fig01/fig1_midd_per_model.csv


,model_id,group,T_original,T_mirror,A,abs_A,S,abs_S,M,actual_reversal,T_negative
0,flow/0000/000,3MIDD_048,-0.001514,0.002210,-0.001862,0.001862,0.000348,0.000348,0.001514,True,-0.007603
1,flow/0000/001,3MIDD_048,-0.002747,0.000110,-0.001428,0.001428,-0.001318,0.001318,0.000110,True,-0.006701
2,flow/0000/002,3MIDD_048,0.000588,0.023965,-0.011689,0.011689,0.012277,0.012277,-0.000588,False,-0.008894
3,flow/0000/003,3MIDD_048,-0.020133,0.040437,-0.030285,0.030285,0.010152,0.010152,0.020133,True,0.001233
4,flow/0000/004,3MIDD_048,-0.016458,-0.017092,0.000317,0.000317,-0.016775,0.016775,-0.016458,False,-0.016692
5,flow/0000/005,3MIDD_048,-0.016402,-0.000890,-0.007756,0.007756,-0.008646,0.008646,-0.000890,False,-0.020464
6,flow/0000/006,3MIDD_048,-0.013238,0.028473,-0.020855,0.020855,0.007618,0.007618,0.013238,True,0.001450
7,flow/0000/007,3MIDD_048,-0.006693,-0.004285,-0.001204,0.001204,-0.005489,0.005489,-0.004285,False,-0.012261
8,flow/0000/008,3MIDD_048,-0.008909,0.003964,-0.006437,0.006437,-0.002472,0.002472,0.003964,True,0.006007
9,flow/0000/009,3MIDD_048,0.001899,0.018121,-0.008111,0.008111,0.010010,0.010010,-0.001899,False,-0.006446


## 6. Adopted validation and controls
Use frozen source records for matched controls, geometry, learning, interventions, and duration. Do not substitute historical intervention estimates for the common-gray operation.


In [6]:
adopted = adopted_analysis(FIGURE, tables)
for name, frame in adopted.items():
    print(name)
    display(frame.head(20))


## 7. Statistics
Model/seed is the unit. Stored intervals retain the original inference choices. Fig3/S7 additionally replay the exact original bootstrap seed and RNG consumption order; cached null draws are not regenerated or tuned.


In [7]:
statistics = statistical_tables(tables)
for name, frame in statistics.items():
    print(name)
    display(frame)


## 8. Figure generation
The shared plotting source preserves the adopted layout and labels; no manual PDF editing.


In [8]:
fig = render(FIGURE)
display(fig)


Saved 1 inputs 3 checks 4


<Figure size 1430x1040 with 5 Axes>

## 9. Numerical summary
Summaries remain labeled by their source; repeated conditions are not independent observations.


In [9]:
print("Figure:", FIGURE, "primary tables:", len(primary), "adopted tables:", len(adopted))


Figure: 1 primary tables: 1 adopted tables: 0


## 10. Reproducibility metadata and hashes
Record inputs, shared source versions, numerical summaries, software, outputs, and limitations.


In [10]:
record = write_reproduction_record(FIGURE, ROOT, manifest, integration_checks, primary, adopted)
print("Canonical notebook:", record["canonical_notebook"])
print("Model inference:", record["inference_performed"])


Canonical notebook: notebooks/fig01/fig_1.ipynb
Model inference: False


## Historical acquisition source (disabled during reproduction)
The cells below document original inference and must not be run during artifact-only figure regeneration. They are preserved for audit.

In [11]:
if False:  # acquisition archive; no new inference
    ## #cell 001
    import os
    import importlib
    import site
    import subprocess
    import sys
    from pathlib import Path
    from typing import List
    
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
    
    if IN_COLAB:
        os.environ["FLYVIS_ROOT_DIR"] = "/content/flyvis_data"
    
        flyvis_checkout = Path("/content/flyvis")
        if not flyvis_checkout.exists():
            subprocess.run(
                ["git", "clone", "--branch", "v1.2.0", "--depth", "1",
                 "--single-branch", "https://github.com/TuragaLab/flyvis.git",
                 str(flyvis_checkout)],
                check=True,
            )
        else:
            if not (flyvis_checkout / "pyproject.toml").is_file():
                raise RuntimeError("/content/flyvis exists but is not a FlyVis checkout.")
            head = subprocess.check_output(
                ["git", "-C", str(flyvis_checkout), "rev-parse", "HEAD"], text=True
            ).strip()
            release = subprocess.check_output(
                ["git", "-C", str(flyvis_checkout), "rev-parse", "v1.2.0^{commit}"], text=True
            ).strip()
            if head != release:
                raise RuntimeError("Existing /content/flyvis is not at v1.2.0. Start a fresh Colab runtime.")
            print("Reusing existing FlyVis v1.2.0 checkout.")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", ".[examples]"],
            cwd=flyvis_checkout,
            check=True,
        )
    
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "flyvis-midd[figures] @ git+https://github.com/poxsalop5029/flyvis-midd.git"],
            capture_output=True,
            text=True,
        )
        if result.returncode:
            raise RuntimeError((result.stderr or result.stdout)[-5000:])
        print("Installed flyvis-midd from the public GitHub repository.")
        REPO_ROOT = Path("/content")
    
        # Editable installs made by pip must also be visible to this running kernel.
        for package_dir in site.getsitepackages():
            site.addsitedir(package_dir)
        # Official examples run inside the checkout; make the same source visible
        # without changing where this notebook saves its figures and data.
        checkout_path = str(flyvis_checkout)
        if checkout_path not in sys.path:
            sys.path.insert(0, checkout_path)
        importlib.invalidate_caches()
        # A previous import from /content can cache the checkout directory as a
        # namespace package before the editable installation becomes visible.
        cached_flyvis = sys.modules.get("flyvis")
        if cached_flyvis is not None and getattr(cached_flyvis, "__file__", None) is None:
            for module_name in list(sys.modules):
                if module_name == "flyvis" or module_name.startswith("flyvis."):
                    del sys.modules[module_name]
    else:
        REPO_ROOT = next(
            path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file()
        )
        sys.path.insert(0, str(REPO_ROOT / "src"))
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    import flyvis
    from flyvis.datasets.rendering import BoxEye
    from flyvis_midd import load_midd_sequences
    
    print(f"FlyVis source: {flyvis.__file__}")
    print(f"FlyVis {flyvis.__version__}; data root: {flyvis.root_dir}")
    plt.rcParams["figure.dpi"] = 150


In [12]:
if False:  # acquisition archive; no new inference
    ## #cell 002
    import numpy as np
    import matplotlib.pyplot as plt
    from flyvis_midd import build_3midd_sequences, build_4midd_sequences, render_sector_image
    
    N_UNITS = 24
    C_3MIDD = 48
    C1_4MIDD, C2_4MIDD = 85, 170
    IMAGE_SIZE = 382
    INNER_DIAMETER = 55
    PHASE_DEG = 0.0
    
    three = build_3midd_sequences(C_3MIDD, n_units=N_UNITS)
    four = build_4midd_sequences(C1_4MIDD, C2_4MIDD, n_units=N_UNITS)
    
    stimuli = [
        ("3MIDD", "Original", three["original"]),
        ("3MIDD", "Mirror", three["mirror"]),
        ("3MIDD", "Negative control", three["negative"]),
        ("4MIDD", "Original", four["original"]),
        ("4MIDD", "Mirror", four["mirror"]),
        ("4MIDD", "Negative control", four["negative"]),
    ]
    
    rendered = {}
    for family, label, sequence in stimuli:
        image, mask = render_sector_image(
            sequence, image_size=IMAGE_SIZE, inner_diameter=INNER_DIAMETER,
            outer_diameter=IMAGE_SIZE, phase_deg=PHASE_DEG,
        )
        rendered[(family, label)] = (sequence, image, mask)


In [13]:
if False:  # acquisition archive; no new inference
    ## #cell 003
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 7.4), facecolor="white")
    for row, family in enumerate(("3MIDD", "4MIDD")):
        for col, label in enumerate(("Original", "Mirror", "Negative control")):
            ax = axes[row, col]
            _, image, _ = rendered[(family, label)]
            ax.imshow(image, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
            ax.set_title(label, fontsize=11)
            ax.set_axis_off()
        axes[row, 0].text(-0.10, 0.5, family, transform=axes[row, 0].transAxes,
                          ha="right", va="center", fontsize=12, fontweight="bold", rotation=90)
    
    fig.suptitle("Static MIDD stimulus examples", fontsize=15, y=0.98)
    fig.text(0.5, 0.025, "Luminance: 0 = black; 255 = white; intermediate levels as indicated by each MIDD family.",
             ha="center", fontsize=9)
    fig.tight_layout(rect=(0.04, 0.055, 1, 0.95))
    
    FIG_DIR = REPO_ROOT / "figures" / "fig01"
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / "fig1A_midd_stimuli.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / "fig1A_midd_stimuli.pdf", bbox_inches="tight")
    plt.show()


In [14]:
if False:  # acquisition archive; no new inference
    ## #cell 004
    DATA_DIR = REPO_ROOT / "data" / "stimuli"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "condition_names": np.array([f"{family}_{label.lower().replace(' ', '_')}" for family, label, _ in stimuli]),
        "n_units": np.array(N_UNITS),
        "image_size": np.array(IMAGE_SIZE),
        "inner_diameter": np.array(INNER_DIAMETER),
        "phase_deg": np.array(PHASE_DEG),
    }
    for (family, label), (sequence, image, mask) in rendered.items():
        key = f"{family.lower()}_{label.lower().replace(' ', '_')}"
        payload[f"{key}_sequence"] = sequence
        payload[f"{key}_image"] = image
        payload[f"{key}_mask"] = mask
    np.savez_compressed(DATA_DIR / "fig01A_midd_stimuli.npz", **payload)
    print("Saved figure files:", FIG_DIR / "fig1A_midd_stimuli.png", "and PDF")
    print("Saved stimulus arrays:", DATA_DIR / "fig01A_midd_stimuli.npz")


In [15]:
if False:  # acquisition archive; no new inference
    ## #cell 005
    condition_names = [
        "3MIDD_048__original", "3MIDD_048__mirror",
        "3MIDD_207__original", "3MIDD_207__mirror",
        "4MIDD_085_170__original", "4MIDD_085_170__mirror",
    ]
    sequences = load_midd_sequences()
    sequences.shape


In [16]:
if False:  # acquisition archive; no new inference
    ## #cell 006
    fig, axes = plt.subplots(2, 3, figsize=(10, 6), constrained_layout=True)
    for ax, condition, sequence in zip(axes.flat, condition_names, sequences):
        ax.imshow(sequence[0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(condition.replace("__", " / "))
        ax.axis("off")
    fig.suptitle("Static MIDD source images / frame 0")
    plt.show()


In [17]:
if False:  # acquisition archive; no new inference
    ## #cell 007
    # Render freshly with the canonical BoxEye; no persistent Directory cache.
    from flyvis_midd.evaluation import CANONICAL_MIDD, analyze_midd_flow, aggregate_midd_metrics
    from flyvis_midd.flow import run_flow_inference
    CONFIG = CANONICAL_MIDD
    eye = BoxEye(extent=CONFIG.eye_extent, kernel_size=CONFIG.eye_kernel_size)
    rendered_sequences = eye(torch.as_tensor(sequences, dtype=torch.float32)).cpu().numpy()
    assert rendered_sequences.shape == (6, CONFIG.n_frames, 1, 721)
    centers_yx = eye.receptor_centers.detach().cpu().numpy()
    rendered_sequences.shape


In [18]:
if False:  # acquisition archive; no new inference
    ## #cell 008
    data = torch.as_tensor(rendered_sequences, dtype=torch.float32)
    assert data.shape == (6, CONFIG.n_frames, 1, 721)
    data[0].shape


In [19]:
if False:  # acquisition archive; no new inference
    ## #cell 009
    # Colab downloads official pretrained data; local data is already available.
    if IN_COLAB:
        !flyvis download-pretrained


In [20]:
if False:  # acquisition archive; no new inference
    ## #cell 010
    # Use explicit official model IDs, fresh modules and a fixed CPU device.
    MODEL_IDS = [f"flow/0000/{i:03d}" for i in range(50)]
    DEVICE = torch.device("cpu")
    model_paths = {mid: flyvis.results_dir / mid for mid in MODEL_IDS}
    if not all(path.is_dir() for path in model_paths.values()):
        raise FileNotFoundError("Run flyvis download-pretrained before inference")
    print("Official networks:", MODEL_IDS[0], "...", MODEL_IDS[-1])


In [21]:
if False:  # acquisition archive; no new inference
    ## #cell 011
    condition_names = [
        "3MIDD_048__original",
        "3MIDD_048__mirror",
        "3MIDD_207__original",
        "3MIDD_207__mirror",
        "4MIDD_085_170__original",
        "4MIDD_085_170__mirror",
    ]
    
    assert len(condition_names) == len(data)


In [22]:
if False:  # acquisition archive; no new inference
    ## #cell 012
    # Same explicit canonical inference path as Fig.2; no EnsembleView simulation.
    import gc, json, hashlib
    negative_image, _ = render_sector_image(
        build_3midd_sequences(48)["negative"], image_size=382,
        inner_diameter=55, outer_diameter=382, phase_deg=0.0,
    )
    negative_movie = np.repeat((negative_image.astype(np.float32)/255)[None, None], CONFIG.n_frames, axis=1)
    negative_input = eye(torch.as_tensor(negative_movie)).cpu()
    inputs = {name: data[[index]] for index, name in enumerate(condition_names)}
    inputs["3MIDD_048__negative"] = negative_input
    flow_lists = {condition: [] for condition in inputs}
    for index, model_id in enumerate(MODEL_IDS, 1):
        view = flyvis.NetworkView(model_paths[model_id])
        network = view.init_network()
        decoder = view.init_decoder()["flow"]
        for condition, movie in inputs.items():
            result = run_flow_inference(movie, network, decoder, dt=CONFIG.dt,
                                        fade=CONFIG.fade_seconds, device=DEVICE)
            flow_lists[condition].append(result["flow"].numpy())
        print(f"[{index:02d}/50] {model_id}: canonical inference complete")
        del result, decoder, network, view
        gc.collect()
    decoded_flow_by_condition = {name: np.stack(flows) for name, flows in flow_lists.items()}
    del flow_lists


In [23]:
if False:  # acquisition archive; no new inference
    ## #cell 013
    # run_flow_inference already canonicalizes layout and returns raw decoder (u,v).
    for condition, flow in decoded_flow_by_condition.items():
        assert flow.shape == (50, 1, CONFIG.n_frames, 2, 721)
        assert np.isfinite(flow).all()
        print(condition, flow.shape)


In [24]:
if False:  # acquisition archive; no new inference
    ## #cell 014
    STABLE = slice(CONFIG.analysis_start, CONFIG.analysis_stop)
    INNER, OUTER = CONFIG.inner_radius_fraction, CONFIG.outer_radius_fraction
    metric_rows = []
    for condition, flows in decoded_flow_by_condition.items():
        group, variant = condition.split("__")
        for index, model_id in enumerate(MODEL_IDS):
            metric = analyze_midd_flow(flows[index], centers_yx, config=CONFIG)
            metric_rows.append({"model_id": model_id, "group": group, "variant": variant,
                                **{key: value for key, value in metric.items() if key != "annulus_mask"}})
    metrics_table = pd.DataFrame(metric_rows)
    midd_rows = []
    for (model_id, group), q in metrics_table.groupby(["model_id", "group"]):
        variant_metrics = q.set_index("variant").to_dict("index")
        midd_rows.append({"model_id": model_id, "group": group, **aggregate_midd_metrics(variant_metrics)})
    midd_table = pd.DataFrame(midd_rows).sort_values(["group", "model_id"])
    fixed = midd_table[midd_table.group.eq("3MIDD_048")].set_index("model_id").loc[MODEL_IDS]
    T_O = fixed.T_original.to_numpy()
    T_M = fixed.T_mirror.to_numpy()
    T_N = fixed.T_negative.to_numpy()
    actual_reversal = fixed.actual_reversal.to_numpy()
    midd_plus = np.flatnonzero(actual_reversal)
    midd_minus = np.flatnonzero(~actual_reversal)
    qa=[]
    def check(name, passed, detail=""):
        qa.append({"check":name,"pass":bool(passed),"detail":str(detail)})
    check("official 50 models and 3 groups",len(midd_table)==150 and set(midd_table.model_id)==set(MODEL_IDS))
    check("finite scalar measurements",np.isfinite(metrics_table.select_dtypes("number").to_numpy()).all())
    check("A/S reconstruction",np.allclose(midd_table.S+midd_table.A,midd_table.T_original) and np.allclose(midd_table.S-midd_table.A,midd_table.T_mirror))
    check("reversal iff positive margin",np.array_equal(midd_table.actual_reversal,midd_table.M>0))
    check("canonical frame and coordinate contract",STABLE==slice(10,50) and CONFIG.flip_decoder_y and INNER==.14 and OUTER==.98)
    for group,count,mean_a in [("3MIDD_048",21,-.00934561245223449),("3MIDD_207",18,.008258561771064806),("4MIDD_085_170",12,.003209041537928414)]:
        q=midd_table[midd_table.group.eq(group)]
        check(group+" regression",int(q.actual_reversal.sum())==count and np.isclose(q.A.mean(),mean_a,atol=1e-6,rtol=0),(int(q.actual_reversal.sum()),q.A.mean()))
        print(group,"reversal",int(q.actual_reversal.sum()),"/50; mean A",q.A.mean())
    # Model-wise comparison to unchanged Fig.2 tables when available locally.
    for group,file,c in [("3MIDD_048","fig2B_luminance_A_per_model.csv",48),("3MIDD_207","fig2B_luminance_A_per_model.csv",207),("4MIDD_085_170","fig2D_4MIDD_A_per_model.csv",None)]:
        path=REPO_ROOT/"data/precomputed/fig02"/file
        if path.is_file():
            source=pd.read_csv(path)
            if c is not None:source=source[source.C.eq(c)]
            q=midd_table[midd_table.group.eq(group)].merge(source,on="model_id",suffixes=("_new","_source"),validate="one_to_one")
            check(group+" Fig.2 per-model regression",len(q)==50 and np.allclose(q.T_original,q.original,atol=1e-6,rtol=0) and np.allclose(q.T_mirror,q.mirror,atol=1e-6,rtol=0) and np.array_equal(q.actual_reversal_new,q.actual_reversal_source))
    OUT = REPO_ROOT/"data/precomputed/fig01"
    OUT.mkdir(parents=True,exist_ok=True)
    metrics_table.to_csv(OUT/"fig1_tangent_per_model_condition.csv",index=False)
    midd_table.to_csv(OUT/"fig1_midd_per_model.csv",index=False)
    qa_table=pd.DataFrame(qa)
    qa_table.to_csv(OUT/"fig1_canonical_QA.csv",index=False)
    (OUT/"metadata.json").write_text(json.dumps({"contract":CONFIG.provenance(),"model_ids":MODEL_IDS,"conditions":list(inputs),"device":str(DEVICE),"flyvis_version":flyvis.__version__,"input_sha256":{name:hashlib.sha256(movie.numpy().tobytes()).hexdigest() for name,movie in inputs.items()}},indent=2)+"\n")
    display(qa_table)
    assert qa_table['pass'].all(), "Canonical regression failed; inspect saved QA."
    print(f"Actual reversal: {len(midd_plus)}/50",midd_plus)


In [25]:
if False:  # acquisition archive; no new inference
    ## #cell 015
    # Representative MIDD flow snapshots
    
    import numpy as np
    import matplotlib.pyplot as plt
    
    from flyvis.datasets.rendering import BoxEye
    from flyvis_midd.plotting import draw_hex_flow
    
    
    # Select two models nearest each group's median flow strength.
    midd_plus = np.flatnonzero(actual_reversal)
    midd_minus = np.flatnonzero(~actual_reversal)
    strength = (np.abs(T_O) + np.abs(T_M)) / 2
    
    def pick_typical(indices, n=2):
        if len(indices) == 0:
            raise ValueError("No models found for one reversal group")
        median = np.median(strength[indices])
        return indices[np.argsort(np.abs(strength[indices] - median))[:n]]
    
    midd_plus_models = pick_typical(midd_plus)
    midd_minus_models = pick_typical(midd_minus)
    representative_models = [*midd_plus_models, *midd_minus_models]
    representative_labels = ["Reversal", "Reversal", "Non-reversal", "Non-reversal"]
    assert np.array_equal(actual_reversal[representative_models], [True, True, False, False])
    representatives = pd.DataFrame({
        "model_id": [MODEL_IDS[i] for i in representative_models],
        "label": representative_labels,
        "actual_reversal": actual_reversal[representative_models],
    })
    representatives.to_csv(OUT / "fig1_representatives.csv", index=False)
    
    
    conditions = [
        "3MIDD_048__original",
        "3MIDD_048__mirror",
    ]
    
    condition_to_index = {
        name: i for i, name in enumerate(condition_names)
    }
    
    eye = BoxEye(extent=15, kernel_size=13)
    centers_yx = eye.receptor_centers.detach().cpu().numpy()
    
    fig, axes = plt.subplots(4, 2, figsize=(6, 10))
    
    for row, (model, label) in enumerate(zip(representative_models, representative_labels)):
        for col, condition in enumerate(conditions):
    
            flow = np.asarray(
                decoded_flow_by_condition[condition][model]
            )
    
            if flow.ndim == 4:
                flow = flow[0]
    
            # flow: (frames, 2, 721)
            frame = flow.shape[0] // 2
    
            flow_uv = flow[frame].T
            # -> (721, 2)
    
            movie_input = np.asarray(
                data[condition_to_index[condition]]
            )
    
            # expected: (frames, 1, 721)
            values = np.squeeze(movie_input[frame])
    
            ax = axes[row, col]
    
            draw_hex_flow(
                ax,
                centers_yx,
                values,
                flow_uv,
                title=(
                    ["Original", "Mirror"][col]
                    if row == 0
                    else ""
                ),
                stride=1,
                axis_off=True,
            )
    
            if col == 0:
                ax.text(
                    -0.08,
                    0.5,
                    f"{label}\nmodel {model:02d}",
                    transform=ax.transAxes,
                    ha="right",
                    va="center",
                )
    
    plt.tight_layout()
    plt.show()


In [26]:
if False:  # acquisition archive; no new inference
    ## #cell 016
    # Propeller positive control: CW_mid, CCW_mid, and Static
    
    import matplotlib.pyplot as plt
    import torch
    import flyvis
    from flyvis.datasets.rendering import BoxEye
    from flyvis_midd import PROPELLER_CONDITIONS, make_propeller_stimulus_set
    from flyvis_midd.flow import run_flow_inference
    from flyvis_midd.plotting import draw_hex_flow
    
    conditions = {name: PROPELLER_CONDITIONS[name] for name in ("CW_mid", "CCW_mid", "Static")}
    stimuli = make_propeller_stimulus_set(conditions, n_frames=50, fps=50, size=128)
    eye = BoxEye(extent=15, kernel_size=13)
    luminance = eye(torch.as_tensor(stimuli, dtype=torch.float32))
    
    view = flyvis.NetworkView(flyvis.results_dir / "flow/0000/000")
    result = run_flow_inference(
        luminance,
        view.init_network(),
        view.init_decoder()["flow"],
        dt=1 / 50,
        fade=1.0,
        device=torch.device("cpu"),
    )
    flow = result["flow"].numpy()
    luminance = result["luminance"].numpy()
    centers_yx = eye.receptor_centers.detach().cpu().numpy()
    frame = flow.shape[1] // 2
    
    fig, axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
    for i, (ax, name) in enumerate(zip(axes, conditions)):
        draw_hex_flow(
            ax,
            centers_yx,
            luminance[i, frame, 0],
            flow[i, frame].T,
            title=name,
            stride=1,
            axis_off=True,
        )
    plt.show()


In [27]:
if False:  # acquisition archive; no new inference
    ## #cell 017
    # Fig. 1C — phenotype frequency across 50 pretrained networks
    
    import matplotlib.pyplot as plt
    
    counts = [len(midd_plus), len(midd_minus)]
    labels = ["Reversal", "Non-reversal"]
    total = sum(counts)
    assert total == 50, f"Expected 50 networks, got {total}"
    percentages = [count / total * 100 for count in counts]
    
    print(f"Total networks: {total}")
    print(f"Reversal: {counts[0]} ({percentages[0]:.1f}%)")
    print(f"Non-reversal: {counts[1]} ({percentages[1]:.1f}%)")
    
    fig, ax = plt.subplots(figsize=(3.4, 2.7))
    bars = ax.bar(labels, counts, color="0.45", width=0.65)
    ax.set_ylabel("Number of networks")
    ax.set_ylim(0, total * 1.18)
    ax.set_title("")
    ax.grid(False)
    for bar, count, percentage in zip(bars, counts, percentages):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + total * 0.025,
            f"{count}/{total}\n{percentage:.0f}%",
            ha="center",
            va="bottom",
        )
    fig.tight_layout()
    plt.show()


In [28]:
if False:  # acquisition archive; no new inference
    ## #cell 018
    # Fig. 1D — model-wise Original vs Mirror tangential flow
    
    import numpy as np
    import matplotlib.pyplot as plt
    
    assert len(T_O) == 50
    assert len(T_M) == 50
    assert len(actual_reversal) == 50
    assert np.isfinite(T_O).all()
    assert np.isfinite(T_M).all()
    assert int(actual_reversal.sum()) == len(midd_plus)
    assert int((~actual_reversal).sum()) == len(midd_minus)
    
    print(f"Networks: {len(T_O)}")
    print(f"Reversal: {len(midd_plus)}")
    print(f"Non-reversal: {len(midd_minus)}")
    
    lim = 1.05 * max(np.max(np.abs(T_O)), np.max(np.abs(T_M)))
    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    ax.scatter(T_O[actual_reversal], T_M[actual_reversal], s=30, color="tab:blue", label="Reversal")
    ax.scatter(T_O[~actual_reversal], T_M[~actual_reversal], s=30, color="tab:orange", label="Non-reversal")
    ax.axhline(0, color="0.65", linewidth=0.8)
    ax.axvline(0, color="0.65", linewidth=0.8)
    ax.plot([-lim, lim], [lim, -lim], linestyle="--", color="0.8", linewidth=0.8)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Original tangential flow (a.u.)")
    ax.set_ylabel("Mirror tangential flow (a.u.)")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()


In [29]:
if False:  # acquisition archive; no new inference
    ## #cell 019
    # Fig. 1D inset — CW-positive tangential-flow geometry
    import numpy as np
    import torch
    import matplotlib.pyplot as plt
    from flyvis_midd.flow import canonical_flow_layout
    from flyvis_midd.plotting import draw_hex_flow
    from flyvis_midd.rotation import rotation_basis
    
    model = int(midd_plus_models[0])
    condition = "3MIDD_048__original"
    flow = canonical_flow_layout(torch.as_tensor(decoded_flow_by_condition[condition][model]))[0].cpu().numpy()
    metric = analyze_midd_flow(decoded_flow_by_condition[condition][model], centers_yx, config=CONFIG)
    _, _, tangent_cw = rotation_basis(centers_yx)  # (x, y-up), same CW basis
    shown_values = np.where(metric["annulus_mask"], 1.0, 0.25)
    shown_tangents = tangent_cw * metric["annulus_mask"][:, None]
    
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    draw_hex_flow(
        ax, centers_yx, shown_values, shown_tangents,
        title="CW-positive tangential-flow definition",
        color="tab:blue", stride=20, size=14, axis_off=True,
    )
    ax.text(0.03, 0.97, "Blue: local CW tangent\nShaded: included annulus", transform=ax.transAxes, ha="left", va="top", color="tab:blue")
    ax.text(
        0.03, 0.04,
        rf"$T=\langle\mathbf{{v}}_i(t)\cdot\hat{{\mathbf{{t}}}}_{{CW,i}}\rangle_{{i\in A,\ t={STABLE.start}\ldots{STABLE.stop-1}}}$" + "\n" + rf"$A: {INNER:.2f}\leq r/r_{{\max}}\leq{OUTER:.2f}$",
        transform=ax.transAxes, ha="left", va="bottom",
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )
    fig.tight_layout()
    plt.show()


In [30]:
if False:  # acquisition archive; no new inference
    ## #cell 020
    # QA only: verify this example uses the same scalar already stored in Fig. 1D.
    assert np.isclose(metric["mean_signed_tangent"], T_O[model], rtol=0, atol=1e-10)
    print(f"Definition check: model {model:02d}, {condition}")
    print("mean signed tangential flow:", metric["mean_signed_tangent"])
    print("annulus receptors:", metric["annulus_receptors"])


In [31]:
if False:  # acquisition archive; no new inference
    ## #cell 021
    # Fig. 1E — Original / Mirror / Negative control
    # Negative T was measured by the same canonical path; it is excluded from A/S/M.
    values = [T_O, T_M, T_N]
    assert all(len(v) == 50 and np.isfinite(v).all() for v in values)
    x = np.arange(3)
    jitter = 0.06 * np.sin(np.arange(50) * 2.4)
    fig, ax = plt.subplots(figsize=(4.2, 3.4))
    for i, row in enumerate(np.column_stack(values)):
        ax.plot(x + jitter[i], row, color="0.75", lw=0.6, zorder=1)
    boxes = ax.boxplot(values, positions=x, widths=0.42, patch_artist=True, showfliers=False, zorder=2)
    colors = ["tab:blue", "tab:orange", "0.55"]
    for box, color in zip(boxes["boxes"], colors):
        box.set_facecolor(color); box.set_alpha(0.22)
    for pos, series, color in zip(x, values, colors):
        ax.scatter(pos + jitter, series, s=12, color=color, alpha=0.75, zorder=3)
    ax.axhline(0, color="0.45", lw=0.8)
    ax.set_xticks(x, ["Original", "Mirror", "Negative"])
    ax.set_ylabel("Mean signed tangential flow (CW positive)")
    ax.set_title("3MIDD C=48 across 50 pretrained networks")
    fig.tight_layout()
    plt.show()


In [32]:
if False:  # acquisition archive; no new inference
    ## #cell 022
    # Assemble the current manuscript-plan Fig. 1 panels A–E from the results above.
    
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle, FancyArrowPatch
    from flyvis_midd.plotting import draw_hex_flow
    
    plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 140})
    fig = plt.figure(figsize=(12, 8.6))
    grid = fig.add_gridspec(2, 3, width_ratios=(1.25, 1, 1), height_ratios=(1.1, 1), hspace=0.25, wspace=0.38)
    
    # A — MIDD stimulus examples
    axes_a = grid[0, 0].subgridspec(2, 3, wspace=0.04, hspace=0.18)
    for r, family in enumerate(("3MIDD", "4MIDD")):
        for c, label in enumerate(("Original", "Mirror", "Negative control")):
            ax = fig.add_subplot(axes_a[r, c])
            _, image, _ = rendered[(family, label)]
            ax.imshow(image, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
            ax.set_title(f"{family}\n{label}", fontsize=7)
            ax.set_axis_off()
            if r == 0 and c == 0:
                ax.text(-0.22, 1.18, "A", transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")
    
    # B — representative Original/Mirror decoded-flow examples
    axes_b = grid[0, 1:].subgridspec(len(representative_models), 2, wspace=0.03, hspace=0.18)
    for row, (model, label) in enumerate(zip(representative_models, representative_labels)):
        for col, condition in enumerate(("3MIDD_048__original", "3MIDD_048__mirror")):
            ax = fig.add_subplot(axes_b[row, col])
            flow = np.asarray(decoded_flow_by_condition[condition][model])
            if flow.ndim == 4:
                flow = flow[0]
            frame = flow.shape[0] // 2
            movie_input = np.asarray(data[condition_to_index[condition]])
            draw_hex_flow(
                ax, centers_yx, np.squeeze(movie_input[frame]), flow[frame].T,
                title=("Original" if col == 0 else "Mirror") if row == 0 else "",
                stride=1, axis_off=True,
            )
            if col == 0:
                ax.text(-0.08, 0.5, f"{label}\nmodel {model:02d}", transform=ax.transAxes,
                        ha="right", va="center", fontsize=7)
            if row == 0 and col == 0:
                ax.text(-0.16, 1.25, "B", transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")
    
    # C — actual-reversal frequency across the official-50 cohort
    ax_c = fig.add_subplot(grid[1, 0])
    counts = [len(midd_plus), len(midd_minus)]
    bars = ax_c.bar(["Actual reversal", "Same sign"], counts, color="0.45", width=0.65)
    ax_c.set_ylabel("Number of models")
    ax_c.set_ylim(0, 50 * 1.18)
    ax_c.tick_params(axis="x", labelrotation=12)
    for bar, count in zip(bars, counts):
        ax_c.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.7, f"{count}/50", ha="center", va="bottom", fontsize=8)
    ax_c.text(-0.14, 1.12, "C", transform=ax_c.transAxes, fontsize=12, fontweight="bold", va="top")
    
    # D — model-wise Original/Mirror tangent; CW-positive schematic is inset
    ax_d = fig.add_subplot(grid[1, 1])
    lim = 1.05 * max(np.max(np.abs(T_O)), np.max(np.abs(T_M)))
    ax_d.scatter(T_O[actual_reversal], T_M[actual_reversal], s=25, color="tab:blue", label="Actual reversal")
    ax_d.scatter(T_O[~actual_reversal], T_M[~actual_reversal], s=25, color="tab:orange", label="Same sign")
    ax_d.axhline(0, color="0.65", linewidth=0.8)
    ax_d.axvline(0, color="0.65", linewidth=0.8)
    ax_d.plot([-lim, lim], [lim, -lim], linestyle="--", color="0.8", linewidth=0.8)
    ax_d.set(xlim=(-lim, lim), ylim=(-lim, lim), aspect="equal", xlabel=r"$T_O$ (CW positive)", ylabel=r"$T_M$ (CW positive)")
    ax_d.legend(frameon=False, fontsize=6, loc="lower left")
    ax_d.text(-0.14, 1.12, "D", transform=ax_d.transAxes, fontsize=12, fontweight="bold", va="top")
    inset = ax_d.inset_axes([0.61, 0.60, 0.34, 0.34])
    inset.add_patch(Circle((0.5, 0.5), 0.32, fill=False, color="0.45", linewidth=0.8))
    inset.add_patch(FancyArrowPatch((0.20, 0.52), (0.78, 0.52), connectionstyle="arc3,rad=0.75",
                                    arrowstyle="-|>", mutation_scale=8, color="tab:blue", linewidth=1.1))
    inset.text(0.5, 0.03, "CW +", ha="center", va="bottom", fontsize=6)
    inset.set(xlim=(0, 1), ylim=(0, 1), aspect="equal")
    inset.set_axis_off()
    
    # E — matched Original/Mirror/Negative population comparison
    ax_e = fig.add_subplot(grid[1, 2])
    values = [T_O, T_M, T_N]
    x = np.arange(3)
    jitter = 0.06 * np.sin(np.arange(50) * 2.4)
    for i, row in enumerate(np.column_stack(values)):
        ax_e.plot(x + jitter[i], row, color="0.75", linewidth=0.6, zorder=1)
    boxes = ax_e.boxplot(values, positions=x, widths=0.42, patch_artist=True, showfliers=False, zorder=2)
    colors = ["tab:blue", "tab:orange", "0.55"]
    for box, color in zip(boxes["boxes"], colors):
        box.set_facecolor(color)
        box.set_alpha(0.22)
    for pos, series, color in zip(x, values, colors):
        ax_e.scatter(pos + jitter, series, s=12, color=color, alpha=0.75, zorder=3)
    ax_e.axhline(0, color="0.45", linewidth=0.8)
    ax_e.set_xticks(x, ["Original", "Mirror", "Negative"])
    ax_e.set_ylabel("Mean signed tangential flow (CW positive)")
    ax_e.text(-0.14, 1.12, "E", transform=ax_e.transAxes, fontsize=12, fontweight="bold", va="top")
    
    fig.subplots_adjust(left=0.09, right=0.99, top=0.98, bottom=0.08)
    FIG_DIR = REPO_ROOT / "figures" / "fig01"
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / "fig01_main.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "fig01_main.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", FIG_DIR / "fig01_main.png", "and", FIG_DIR / "fig01_main.pdf")
